In [2]:
import os
import re
import pandas as pd
from dotenv import load_dotenv
from opendartreader import OpenDartReader
from bs4 import BeautifulSoup

# API 키 로드 및 DART 객체 생성
load_dotenv()
api_key = os.environ.get('DART_API_KEY')
dart = OpenDartReader(api_key)

print("⏳ 데이터 수집 준비 완료...")

# 테스트용 기업(예: 카카오)의 최근 공시 목록 불러오기
# (카카오 고유번호: 00258801)
corp_code = '00258801'
disclosures = dart.list(corp_code, start='20240101') # 2024년 이후 공시

if not disclosures.empty:
    # 가장 최근 공시의 접수번호(rcept_no)와 보고서명(report_nm) 추출
    target_rcept_no = disclosures['rcept_no'].iloc[0]
    target_report_nm = disclosures['report_nm'].iloc[0]
    
    print(f"\n🎯 [타겟 공시 발견]")
    print(f"보고서명: {target_report_nm}")
    print(f"접수번호: {target_rcept_no}")
    
    # OpenDART API를 통해 공시 원문(XML/HTML) 다운로드
    print("\n📥 DART 서버에서 원문 데이터를 긁어오는 중...")
    raw_xml = dart.document(target_rcept_no)
    
    # 전처리 (Data Cleansing)
    # BeautifulSoup으로 HTML 태그 걷어내기
    soup = BeautifulSoup(raw_xml, 'xml')
    clean_text = soup.get_text(separator=' ', strip=True)
    
    # 정규표현식(Regex)으로 불필요한 연속 공백, 특수문자 일부 제거
    clean_text = re.sub(r'\s+', ' ', clean_text) # 띄어쓰기 여러 개를 하나로 통일
    
    # 결과 확인
    print("\n✨ [전처리 완료된 순수 텍스트 미리보기]")
    print("-" * 50)
    print(clean_text[:500] + "\n... (이하 생략)")
    print("-" * 50)
    print(f"📊 총 텍스트 길이: {len(clean_text)} 글자")

else:
    print("❌ 해당 기간에 공시가 없습니다.")

⏳ 데이터 수집 준비 완료...

🎯 [타겟 공시 발견]
보고서명: 대규모기업집단현황공시[분기별공시(대표회사용)]
접수번호: 20260826000632

📥 DART 서버에서 원문 데이터를 긁어오는 중...

✨ [전처리 완료된 순수 텍스트 미리보기]
--------------------------------------------------
기업집단현황공시(분기-대표회사용) 6.2 카카오 대규모기업집단 현황 공시 기업집단명 : 카 카 오 기업집단 동일인 : 김 범 수 기업집단 대표회사 : (주)카카오 작성회사 : (주)카카오 담당자 : 정 현 진 (수석) , 전화번호(1577-3754) 4. 순환출자 현황[ESG] (2) 국내 계열회사간 순환출자 변동 내역[ESG] : 국내 계열회사간 분기별 순환출자 현황 및 변동내역 : 해당사항 없음. 6. 금융ㆍ보험사 의결권 행사 현황[ESG] (1) 금융ㆍ보험사의 국내계열회사주식에 대한 의결권 행사 현황[ESG] (직전 분기 개시일∼종료일 기준, 단위: 주, %) 소속회사명 피출자회사 현황 출자현황 의결권행사 현황 회사명 상장 여부 전체 주식수 주식수 승인 주식수 지분율 주총일 안건 의결권 행사여부 (주)카카오페이 (주)링키지랩 - 69,090 10,363 - 15.00 - 개최실적 없음 - (주)케이큐브홀딩스 (주)카카오 상장 442,981,070 46,253,222 - 10.4
... (이하 생략)
--------------------------------------------------
📊 총 텍스트 길이: 3669 글자
